# Cavity LDOS Bounds for Lossless Systems
Lossless, zero-bandwidth systems require special care for finding an initial point for the dual optimization. Eventually, this will be supported natively in Dolphindes. For now, this is an experimental notebook that shows how to do this for LDOS enhancement using a MOSEK feasibility problem. The theory remains unpublished, but will be updated here when it is.

Note 1: cvxpy + MOSEK are not required to intsall Dolphindes to keep it lightweight. If you'd like to run this notebook, you will need to install these into your environment.
Note 2: This does not currently support quadratic objectives or general constraints.

This notebook is a work in progress and is experimental.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from dolphindes import geometry, photonics
from dolphindes.cvxopt import gcd, OptimizationHyperparameters
from dolphindes.maxwell import plot_real_polar_field, plot_cplx_polar_field, expand_symmetric_field, TM_Polar_FDFD

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Problem Setup

In [ ]:
chi = 5
wvlgth = 1.0
Qabs = np.inf
omega = 2*np.pi / wvlgth * (1 + 1j/2/Qabs)

R_nonpml = 3.5 # Total R
w_pml = 0.5 # surrounding pml thickness
R_tot = R_nonpml + w_pml # total computational domain radius
R_i = 0.02 # inner radius of the cavity
R_o = 1.0 # outer radius of the cavity
assert R_o < R_nonpml, "Outer radius must be within non-PML region"

gpr = 30
dr = 1.0/gpr
Nr = int(np.round(R_tot / dr))
Npml = int(np.round(w_pml / dr))
Nr_i = int(np.round(R_i / dr))
Nr_o = int(np.round(R_o / dr))
Nphi = 108  # if Nphi = n_sectors = 1, we enforce cylindrical symmetry
n_sectors = 6
Nphi_sector = int(Nphi/n_sectors)

assert Nphi % n_sectors == 0, "Nphi must be divisible by n_sectors"

# If the problem is small enough, we can solve the full local problem.
# Otherwise we will run GCD.
SOLVE_LOCAL = False

In [ ]:
# Normalize source so integrated current in the central pixel = 1
J_r = np.zeros(Nr)
J_r[0] = 1.0 / (np.pi * dr**2)  # central pixel current density = 1/area
J = np.kron(np.ones(Nphi_sector), J_r) 
phi_grid = np.linspace(0, 2*np.pi, Nphi, endpoint=False)
r_grid = (np.arange(Nr) + 0.5) * dr

des_mask_polar_sector = np.zeros((Nr, Nphi_sector), dtype=bool)
for i, r in enumerate(r_grid):
    if R_i <= r <= R_o:
        des_mask_polar_sector[i, :] = True
ndof = np.sum(des_mask_polar_sector)

# 2. Instantiate the geometry using Nphi_sector
geo_polar = geometry.PolarFDFDGeometry(Nphi_sector, Nr, Npml, dr, n_sectors=n_sectors)
solver = TM_Polar_FDFD(omega, geo_polar)
Ei_sector = solver.get_TM_field(J)
G0 = solver.get_TM_Gba(des_mask_polar_sector, des_mask_polar_sector)
U = np.eye(ndof) * chi**(-1) - G0

mask_1d = des_mask_polar_sector.flatten(order="F")
A_full_sector = geo_polar.get_pixel_areas()
A_des = A_full_sector[mask_1d]

W_diag = np.sqrt(A_des)
Winv_diag = 1.0 / W_diag

# 3. Symmetrize the Green's function and U matrix
# The photonics class does this by default but here we are explicitly building these matrices in a lightweight manner
# <P|W @ G0 @ W_inv|P> gives <P|G0_{analytical} A|P>, which is what we want, but symmetrized.
# The chi term does not need this since it cancels out.
# We do not need to scale the incident field since we will be doing it when we build the photonics problem later.
G0_sym = G0 * W_diag[:, None] * Winv_diag[None, :]
U_sym = np.eye(ndof) * chi**(-1) - G0_sym
assert np.allclose(U_sym, U_sym.T, atol=1e-10), "U_sym is not strictly symmetric."
print("condition number of U_sym:", np.linalg.cond(U_sym))

condition number of U_sym: 533.4755502545198


In [ ]:
print(f"ndof: {ndof}")
LDOS_vac = -0.5 * np.real(n_sectors * np.sum(np.conj(J) * Ei_sector * geo_polar.get_pixel_areas()))
print(f"Vacuum LDOS (total): {LDOS_vac:.4f}")

ndof: 522
Vacuum LDOS (total): 0.7893


In [ ]:
import cvxpy as cp

epsilon = 1e-6 

# 1. Define decision variables
m = cp.Variable(ndof, complex=True)

# 2. Construct the Hermitian constraint operator
A = cp.multiply(cp.conj(m)[:, None], U_sym)
H = 0.5j * (A.H - A)

# 3. Formulate the strict interior constraint
constraints = [H >> epsilon * np.eye(ndof)]

objective = cp.Minimize(0)  # pure feasibility objective
prob = cp.Problem(objective, constraints)

# 4. Solve
prob.solve(solver=cp.MOSEK, verbose=True)

if prob.status not in ["optimal", "optimal_inaccurate"]:
    raise ValueError(f"Feasibility search failed. Status: {prob.status}")

m_start = m.value

(CVXPY) May 25 03:45:09 PM: Your problem has 522 variables, 272484 constraints, and 0 parameters.
(CVXPY) May 25 03:45:09 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) May 25 03:45:09 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) May 25 03:45:09 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) May 25 03:45:09 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) May 25 03:45:09 PM: Compiling problem (target solver=MOSEK).
(CVXPY) May 25 03:45:09 PM: Reduction chain: Complex2Real -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> MOSEK
(CVXPY) May 25 03:45:09 PM: Applying reduction Complex2Real
(CVXPY) May 25 03:45:09 PM: Applying reduction Dcp2Cone


(CVXPY) May 25 03:45:09 PM: Applying reduction CvxAttr2Constr
(CVXPY) May 25 03:45:09 PM: Applying reduction ConeMatrixStuffing


                                     CVXPY                                     
                                     v1.7.5                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) May 25 03:45:12 PM: Applying reduction MOSEK
(CVXPY) May 25 03:45:15 PM: Finished problem compilation (took 5.720e+00 seconds).
(CVXPY) May 25 03:45:15 PM: Invoking solver MOSEK  to obtain a solution.
(CVXPY) May 25 03:45:15 PM: Problem


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------




(CVXPY) May 25 03:45:15 PM:   Name                   :                 
(CVXPY) May 25 03:45:15 PM:   Objective sense        : maximize        
(CVXPY) May 25 03:45:15 PM:   Type                   : CONIC (conic optimization problem)
(CVXPY) May 25 03:45:15 PM:   Constraints            : 1044            
(CVXPY) May 25 03:45:15 PM:   Affine conic cons.     : 0               
(CVXPY) May 25 03:45:15 PM:   Disjunctive cons.      : 0               
(CVXPY) May 25 03:45:15 PM:   Cones                  : 0               
(CVXPY) May 25 03:45:15 PM:   Scalar variables       : 0               
(CVXPY) May 25 03:45:15 PM:   Matrix variables       : 1 (scalarized: 545490)
(CVXPY) May 25 03:45:15 PM:   Integer variables      : 0               
(CVXPY) May 25 03:45:15 PM: 
(CVXPY) May 25 03:45:15 PM: Optimizer started.
(CVXPY) May 25 03:45:15 PM: Presolve started.
(CVXPY) May 25 03:45:15 PM: Linear dependency checker started.
(CVXPY) May 25 03:45:15 PM: Linear dependency checker terminated.
(CVXP

-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


In [ ]:
ei_des_local = Ei_sector[des_mask_polar_sector.flatten(order="F")]

lossless_local = photonics.Photonics_TM_FDFD(
    omega,
    geo_polar,
    chi,
    des_mask_polar_sector,
    J,
    sparseQCQP=True,
 )

A0_local = sp.csc_array((ndof, ndof), dtype=complex)
s0_local = -0.25j * omega * ei_des_local.conj()
lossless_local.set_objective(
    A0=A0_local,
    s0=s0_local,
    c0=LDOS_vac,
    denseToSparse=True,
 )
lossless_local.setup_QCQP(Pdiags="local", verbose=1)

local_lags = np.concatenate([-m_start.imag, -m_start.real])
lossless_local.QCQP.current_lags = local_lags

A_probe = np.diag(np.conj(m_start)) @ U_sym
H_direct = 0.5j * (A_probe.conj().T - A_probe)
local_feasible = lossless_local.QCQP.is_dual_feasible(local_lags)

assert local_feasible

print("Built the sparse local photonics problem")
print("local_qcqp_type =", type(lossless_local.QCQP).__name__)
print("current_lags.shape =", lossless_local.QCQP.current_lags.shape)
print("local_feasible =", local_feasible)

Precomputed 1044 A matrices and Fs vectors.
Built the sparse local photonics problem
local_qcqp_type = SparseSharedProjQCQP
current_lags.shape = (1044,)
local_feasible = True


In [ ]:
if SOLVE_LOCAL:
    solution = lossless_local.solve_current_dual_problem(method='newton')

    opt_params = OptimizationHyperparameters(
        opttol = 1e-4,
        gradConverge = True,
        min_inner_iter = 5,
        max_restart = np.inf,
        penalty_ratio = 1e-2,
        penalty_reduction = 0.1,
        break_iter_period = 50,
        verbose = 0
    )
else:
    gcd_params = gcd.GCDHyperparameters(
        max_proj_cstrt_num=8,
        orthonormalize=True,
        opt_params=None,
        gcd_tol=1e-3,
    )

    gcd.merge_lead_constraints(lossless_local.QCQP, merged_num=np.inf)
    solution = lossless_local.QCQP.run_gcd(gcd_params=gcd_params)

Precomputed 1 A matrices and Fs vectors.
At GCD iteration #1, best dual bound found is             28851743.77130631.


/home/alessio/code/good_code/dolphindes/dolphindes/cvxopt/qcqp.py:178: CholmodTypeConversionWarning: array contains 64 bit integers; but 32 bit integers are needed; slowing down due to converting
  self.Acho.cholesky_inplace(A)
/home/alessio/code/good_code/dolphindes/dolphindes/cvxopt/qcqp.py:216: CholmodTypeConversionWarning: array contains 64 bit integers; but 32 bit integers are needed; slowing down due to converting
  tmp = self.Acho.cholesky(A)


At GCD iteration #2, best dual bound found is             7813101.563035918.
At GCD iteration #3, best dual bound found is             2194298.6474413467.
At GCD iteration #4, best dual bound found is             2191753.2711021076.
At GCD iteration #5, best dual bound found is             1898214.928387576.
At GCD iteration #6, best dual bound found is             1895373.8034333086.
At GCD iteration #7, best dual bound found is             1887108.5754437689.
At GCD iteration #8, best dual bound found is             1869537.922667733.
At GCD iteration #9, best dual bound found is             1841122.0430071955.
At GCD iteration #10, best dual bound found is             1808347.3157625599.
At GCD iteration #11, best dual bound found is             1803722.0160529506.
At GCD iteration #12, best dual bound found is             1772975.6827464176.
At GCD iteration #13, best dual bound found is             1767842.7126624824.
At GCD iteration #14, best dual bound found is             1740

In [ ]:
lossless_local.QCQP.current_dual / LDOS_vac

np.float64(2205067.2530000056)